In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

In [ ]:
def min_semivar(prices: pd.DataFrame, umbral: float = 0) -> np.ndarray:
    OPTIMIZATION_TOL = 1e-50

    returns = prices.pct_change().dropna()
    n_assets = len(returns.columns)
    corr = returns.corr()

    below_threshold = returns[returns < umbral].fillna(0)
    downside_risk = np.array(below_threshold.std())
    semivar_matrix = (
        downside_risk.reshape(n_assets, 1)
        @ downside_risk.reshape(1, n_assets)
        * corr
    )

    opt = minimize(
        fun=lambda weight: weight.T @ semivar_matrix @ weight,
        x0=np.ones(n_assets) / n_assets,
        bounds=[(0, 1)] * n_assets,
        constraints={"fun": lambda weight: weight @ np.ones(n_assets) - 1, "type": "eq"},
        tol=OPTIMIZATION_TOL,
    )
    
    print(opt)
    return opt.x


def min_variance(
    prices: pd.DataFrame,
    minimum_return: float | None = None,
) -> np.ndarray:
    returns = prices.pct_change().dropna()
    n_assets = len(returns.columns)

    expected_returns = returns.mean().to_numpy(dtype=float) * 252.0
    covariance_matrix = returns.cov().to_numpy(dtype=float) * 252.0

    constraints = [
        {"type": "eq", "fun": lambda weights: np.sum(weights) - 1.0}
    ]

    if minimum_return is not None:
        constraints.append(
            {
                "type": "ineq",
                "fun": lambda weights, mu=expected_returns, target=float(minimum_return): (
                    weights @ mu
                ) - target,
            }
        )

    opt = minimize(
        fun=lambda weights, cov=covariance_matrix: float(weights.T @ cov @ weights),
        x0=np.ones(n_assets) / n_assets,
        method="SLSQP",
        bounds=[(0.0, 1.0)] * n_assets,
        constraints=constraints,
        options={
            "maxiter": 500,
            "ftol": 1e-9,
            "disp": False,
        },
    )

    return opt.x

def max_sharpe(
    prices: pd.DataFrame,
    risk_free_rate: float = 0.0,
) -> np.ndarray:
    returns = prices.pct_change().dropna()
    n_assets = len(returns.columns)

    expected_returns = returns.mean().to_numpy(dtype=float) * 252.0
    covariance_matrix = returns.cov().to_numpy(dtype=float) * 252.0

    def negative_sharpe(weights: np.ndarray) -> float:
        portfolio_variance = float(weights.T @ covariance_matrix @ weights)
        portfolio_variance = max(portfolio_variance, 0.0)
        portfolio_volatility = float(np.sqrt(portfolio_variance))

        if np.isclose(portfolio_volatility, 0.0):
            return float("inf")

        portfolio_return = float(weights @ expected_returns)
        sharpe_ratio = (portfolio_return - risk_free_rate) / portfolio_volatility

        return float(-sharpe_ratio)

    opt = minimize(
        fun=negative_sharpe,
        x0=np.ones(n_assets) / n_assets,
        method="SLSQP",
        bounds=[(0.0, 1.0)] * n_assets,
        constraints=[{"type": "eq", "fun": lambda weights: np.sum(weights) - 1.0}],
        options={
            "maxiter": 500,
            "ftol": 1e-9,
            "disp": False,
        },
    )

    return opt.x
